##### Ocean Trend Analysis (SSH & SST)

**Overview**   
Pixel-wise trend analysis of sea surface temperature (SST) and sea surface height (SSH) using HAC-corrected regression. The analysis focuses on long-term climate variability in the Southwest Indian Ocean.  

**Data**  
- Source: Copernicus Marine Service  
- SST: https://data.marine.copernicus.eu/viewer/expert?view=dataset&dataset=SST_GLO_SST_L4_REP_OBSERVATIONS_010_024  
- SSH: https://data.marine.copernicus.eu/viewer/expert?view=dataset&dataset=SEALEVEL_GLO_PHY_CLIMATE_L4_MY_008_057  

**Domain**  
- Region: 10°–30°S, 30°–50°E  
- Time range: 1993 – 2025  

**Methods**  
- Pixel-wise linear regression  
- HAC (heteroskedasticity and autocorrelation consistent) correction  
- Time-series analysis  

**Tools**  
Python (NumPy, xarray, Matplotlib), NetCDF

**Context**  
Analysis contributed to climate reporting work with the Norwegian Meteorological Institute and Instituto Nacional de Meteorologia (Mozambique).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import statsmodels.api as sm        
from tqdm import tqdm                

In [ ]:
# Define paths for the data (MODIFY AS DESIRED)
# NOTE!!!! In my case I had to download the timeseries in parts (3 years at a time) because of the copernicus marine data service 
# limitations. If the dataset is already in one file modify code accordingly.
path_ssh = r'C:\Users\data\data.nc'
psst1 = r'D:\data\sst9395.nc'
psst2 = r'D:\data\sst9698.nc'
psst3 = r'D:\data\sst9901.nc'
psst4 = r'D:\data\sst0204.nc'
psst5 = r'D:\data\sst0507.nc'
psst6 = r'D:\data\sst0810.nc'
psst7 = r'D:\data\sst1113.nc'
psst8 = r'D:\data\sst1416.nc'
psst9 = r'D:\data\sst1719.nc'
psst10 = r'D:\data\sst2022.nc'
psst11 = r'D:\data\sst2325.nc'

# open and assign the datasets to variables
ssh = xr.open_dataset(path_ssh)
sst1 = xr.open_dataset(psst1)
sst2 = xr.open_dataset(psst2)
sst3 = xr.open_dataset(psst3)
sst4 = xr.open_dataset(psst4)
sst5 = xr.open_dataset(psst5)
sst6 = xr.open_dataset(psst6)
sst7 = xr.open_dataset(psst7)
sst8 = xr.open_dataset(psst8)
sst9 = xr.open_dataset(psst9)
sst10 = xr.open_dataset(psst10)
sst11= xr.open_dataset(psst11)

# Combine sst datasets into one 
datasets = [sst1, sst2, sst3, sst4, sst5, sst6, sst7, sst8, sst9, sst10, sst11]
sst_all = xr.concat(datasets, dim="time")

In [ ]:
def pixel_trend_ssh(ds,var_name,extent,time_dim="time",lat_dim="latitude",lon_dim="longitude",min_fraction_valid=0.7,hac_lags=365):
    """
    Computes pixel-wise linear sea level trends and their statistical significance
    from a gridded sea level dataset and produces a spatial map of the results.

    Trends are estimated using ordinary least squares regression on time expressed
    in fractional years, with statistical significance assessed using
    heteroskedasticity- and autocorrelation-consistent (HAC / Newey–West)
    standard errors.

    Parameters:
    -----------
    ds : xarray.Dataset
        Dataset containing the sea level variable with dimensions (time, latitude, longitude).
    var_name : str
        Name of the sea level variable in the dataset.
    extent : list
        Plotting extent in the format [lon_min, lon_max, lat_min, lat_max].
    time_dim : str, optional
        Name of the time dimension. Default is 'time'.
    lat_dim : str, optional
        Name of the latitude dimension. Default is 'latitude'.
    lon_dim : str, optional
        Name of the longitude dimension. Default is 'longitude'.
    min_fraction_valid : float, optional
        Minimum fraction of valid (non-NaN) observations required for a pixel to be included in the trend analysis. Default is 0.7.
    hac_lags : int, optional
        Maximum lag (in time steps) used for HAC (Newey–West) correction of standard errors. Default is 365, it accounts for the full seasonal cycle.

    Returns:
    --------
    trend : np.ndarray
        Two-dimensional array (latitude × longitude) of linear sea level trends, expressed in mm/year.
    pval : np.ndarray
        Two-dimensional array (latitude × longitude) of HAC-corrected p-values associated with the linear trends.

    Functionality:
    --------------
    - Extracts the specified sea level variable from the dataset.
    - Converts time coordinates to fractional years for physically meaningful trend estimation.
    - Fits a linear regression model independently at each grid point.
    - Accounts for temporal autocorrelation using HAC (Newey–West) corrected standard errors.
    - Applies a minimum data-coverage criterion to exclude poorly sampled pixels.
    - Produces spatial map of sea level trends.
    - Saves the resulting figure to disk.

    Notes:
    ------
    - Trends represent the mean linear rate of sea level change over the analysis period.
    - P-values quantify the statistical significance of the linear trend and account for temporal autocorrelation in the time series.
    - Results are intended for large-scale, long-term trend assessment rather than short-term variability analysis.
    """

    # ---------------------------------------------------------------------
    # 1. Extract the target variable
    # ---------------------------------------------------------------------
    da = ds[var_name]                                 # DataArray: (time, lat, lon)

    # ---------------------------------------------------------------------
    # 2. Convert time to fractional years 
    # ---------------------------------------------------------------------
    time = da[time_dim]                                                       # Time coordinate
    time_years = (time.dt.year+ (time.dt.dayofyear - 1) / 365.25).values      # Fractional years as NumPy array

    # Add intercept term for regression
    X = sm.add_constant(time_years)                   # Shape: (time, 2)

    # ---------------------------------------------------------------------
    # 3. Prepare output arrays
    # ---------------------------------------------------------------------
    nlat = da.sizes[lat_dim]                          # Number of latitude points
    nlon = da.sizes[lon_dim]                          # Number of longitude points

    trend = np.full((nlat, nlon), np.nan)             # Trend in mm/year
    pval = np.full((nlat, nlon), np.nan)              # HAC-corrected p-values

    # ---------------------------------------------------------------------
    # 4. Loop over spatial grid
    # ---------------------------------------------------------------------
    for i in tqdm(range(nlat), desc="Processing latitude"):
        for j in range(nlon):
            # Extract time series for this pixel
            y = da[:, i, j].values                    # Sea level time series
            # Identify valid (non-NaN) observations
            valid = np.isfinite(y)
            # Require sufficient temporal coverage
            if np.sum(valid) < min_fraction_valid * len(y):
                continue                              # Skip poorly sampled pixels
            # Subset data to valid points only
            y_valid = y[valid]
            X_valid = X[valid, :]

            # -----------------------------------------------------------------
            # 5. Fit OLS model
            # -----------------------------------------------------------------
            model = sm.OLS(y_valid, X_valid)
            results = model.fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})        # Correct for autocorrelation

            # -----------------------------------------------------------------
            # 6. Store slope and p-value
            # -----------------------------------------------------------------
            trend[i, j] = results.params[1] * 1000    # Convert m/year → mm/year
            pval[i, j] = results.pvalues[1]           # HAC-corrected p-value

    # ---------------------------------------------------------------------
    # 7. Plotting 
    # ---------------------------------------------------------------------
    lon = da[lon_dim].values
    lat = da[lat_dim].values

    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

    # ---------------- Trend map ----------------
    im = ax.pcolormesh(lon, lat, trend, transform=ccrs.PlateCarree(), cmap="YlGnBu")

    ax.set_title("Nível do Mar (1993 - 2025)", fontsize=15, fontweight="bold")
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    ax.set_extent(extent)
    ax.coastlines()
    gl = ax.gridlines(draw_labels=True)
    gl.right_labels = False
    gl.top_labels = False
    gl.xlabel_style = {"size": 12}
    gl.ylabel_style = {"size": 12}
    ax.add_feature(cfeature.LAND, zorder=1, edgecolor="black")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Tendência (mm/Ano)", fontsize=15)
    cbar.ax.tick_params(labelsize=12)

    plt.tight_layout()
    plt.savefig("out/ssh_pixel_trend.png", dpi=300)
    plt.close()

    # ---------------------------------------------------------------------
    # 8. Return numerical results for reproducibility
    # ---------------------------------------------------------------------
    return trend, pval


In [ ]:
def pixel_trend_sst(ds,var_name,extent,time_dim="time",lat_dim="latitude",lon_dim="longitude",min_fraction_valid=0.7,hac_lags=365):
    """
    Computes pixel-wise linear sea surface temperature trends and their statistical significance
    from a gridded sst dataset and produces a spatial map of the results.

    Trends are estimated using ordinary least squares regression on time expressed
    in fractional years, with statistical significance assessed using
    heteroskedasticity- and autocorrelation-consistent (HAC / Newey–West)
    standard errors.

    Parameters:
    -----------
    ds : xarray.Dataset
        Dataset containing the SST variable with dimensions (time, latitude, longitude).
    var_name : str
        Name of the sst variable in the dataset.
    extent : list
        Plotting extent in the format [lon_min, lon_max, lat_min, lat_max].
    time_dim : str, optional
        Name of the time dimension. Default is 'time'.
    lat_dim : str, optional
        Name of the latitude dimension. Default is 'latitude'.
    lon_dim : str, optional
        Name of the longitude dimension. Default is 'longitude'.
    min_fraction_valid : float, optional
        Minimum fraction of valid (non-NaN) observations required for a pixel to be included in the trend analysis. Default is 0.7.
    hac_lags : int, optional
        Maximum lag (in time steps) used for HAC (Newey–West) correction of standard errors. Default is 365, it accounts for the full seasonal cycle.

    Returns:
    --------
    trend : np.ndarray
        Two-dimensional array (latitude × longitude) of linear sst trends, expressed in °C/year.
    pval : np.ndarray
        Two-dimensional array (latitude × longitude) of HAC-corrected p-values associated with the linear trends.

    Functionality:
    --------------
    - Extracts the specified sst variable from the dataset.
    - Converts time coordinates to fractional years for physically meaningful trend estimation.
    - Fits a linear regression model independently at each grid point.
    - Accounts for temporal autocorrelation using HAC (Newey–West) corrected standard errors.
    - Applies a minimum data-coverage criterion to exclude poorly sampled pixels.
    - Produces a spatial map of sst trends.
    - Saves the resulting figure to disk.

    Notes:
    ------
    - Trends represent the mean linear rate of sst change over the analysis period.
    - P-values quantify the statistical significance of the linear trend and account for temporal autocorrelation in the time series.
    - Results are intended for large-scale, long-term trend assessment rather than short-term variability analysis.
    """

    # ---------------------------------------------------------------------
    # 1. Extract the target variable
    # ---------------------------------------------------------------------
    da = ds[var_name] - 273.15                             # DataArray: (time, lat, lon), convert from Kelvin to Celsius
    
    # ---------------------------------------------------------------------
    # 2. Convert time to fractional years 
    # ---------------------------------------------------------------------
    time = da[time_dim]                                                       # Time coordinate
    time_years = (time.dt.year+ (time.dt.dayofyear - 1) / 365.25).values      # Fractional years as NumPy array

    # Add intercept term for regression
    X = sm.add_constant(time_years)                   # Shape: (time, 2)

    # ---------------------------------------------------------------------
    # 3. Prepare output arrays
    # ---------------------------------------------------------------------
    nlat = da.sizes[lat_dim]                          # Number of latitude points
    nlon = da.sizes[lon_dim]                          # Number of longitude points

    trend = np.full((nlat, nlon), np.nan)             # Trend in °C/year
    pval = np.full((nlat, nlon), np.nan)              # HAC-corrected p-values

    # ---------------------------------------------------------------------
    # 4. Loop over spatial grid
    # ---------------------------------------------------------------------
    for i in tqdm(range(nlat), desc="Processing latitude"):
        for j in range(nlon):
            # Extract time series for this pixel
            y = da[:, i, j].values                    # Sea surface temperature time series
            # Identify valid (non-NaN) observations
            valid = np.isfinite(y)
            # Require sufficient temporal coverage
            if np.sum(valid) < min_fraction_valid * len(y):
                continue                              # Skip poorly sampled pixels
            # Subset data to valid points only
            y_valid = y[valid]
            X_valid = X[valid, :]

            # -----------------------------------------------------------------
            # 5. Fit OLS model
            # -----------------------------------------------------------------
            model = sm.OLS(y_valid, X_valid)
            results = model.fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})        # Correct for autocorrelation

            # -----------------------------------------------------------------
            # 6. Store slope and p-value
            # -----------------------------------------------------------------
            trend[i, j] = results.params[1]            # sst slope (°C/year)
            pval[i, j] = results.pvalues[1]            # HAC-corrected p-value

    # ---------------------------------------------------------------------
    # 7. Plotting 
    # ---------------------------------------------------------------------
    lon = da[lon_dim].values
    lat = da[lat_dim].values

    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

    # ---------------- Trend map ----------------
    im = ax.pcolormesh(lon, lat, trend, transform=ccrs.PlateCarree(), cmap="YlOrRd")

    ax.set_title("Temperatura da Superfície do Mar (1993 - 2025)", fontsize=15, fontweight="bold")
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    ax.set_extent(extent)
    ax.coastlines()
    gl = ax.gridlines(draw_labels=True)
    gl.right_labels = False
    gl.top_labels = False
    gl.xlabel_style = {"size": 12}
    gl.ylabel_style = {"size": 12}
    ax.add_feature(cfeature.LAND, zorder=1, edgecolor="black")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Tendência (°C/Ano)", fontsize=15)
    cbar.ax.tick_params(labelsize=12)

    plt.tight_layout()
    plt.savefig("out/ssh_pixel_trend.png", dpi=300)
    plt.close()    

    # ---------------------------------------------------------------------
    # 8. Return numerical results for reproducibility
    # ---------------------------------------------------------------------
    return trend, pval          


In [ ]:
# Run the function for ssh.
# NOTE!!! an "out" folder should be created in the same directory as the notebook.
pixel_trend_ssh(ssh,'adt',[30, 50, -10, -30])

In [ ]:
# Run the function for ssh.
# NOTE!!! an "out" folder should be created in the same directory as the notebook.
pixel_trend_sst(sst_all,'analysed_sst',[30, 50, -10, -30])